<a href="https://colab.research.google.com/github/danieligelnik/CCFraudProject/blob/main/DS_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from sklearn.model_selection import train_test_split

In [ ]:
!pip install import-ipynb
import import_ipynb
import requests
import os

# 1. URL to the RAW version of the notebook on GitHub
github_url = "https://raw.githubusercontent.com/danieligelnik/CCFraudProject/main/help_functions.ipynb"
# Corrected the typo in the filename here:
notebook_filename = "help_functions.ipynb"

# 2. Download the notebook file locally
response = requests.get(github_url)
with open(notebook_filename, 'wb') as f:
    f.write(response.content)

# 3. Import the notebook as a module
try:
    # The module name must match the filename (without .ipynb)
    from help_functions import *
    #from help_functions import load_kagglehub_dataset
    print(f"Successfully imported functions from {notebook_filename}")
except Exception as e:
    print(f"Error importing notebook: {e}")

# **Data**

In [ ]:
# Set the path to the file you'd like to load
credit_cards_path_test = "fraudTest.csv"
credit_cards_path_train = "fraudTrain.csv"
main_path="dermisfit/fraud-transactions-dataset"

# Load the latest version
df_cards_train = load_kagglehub_dataset(main_path, credit_cards_path_train)
df_cards_test = load_kagglehub_dataset(main_path, credit_cards_path_test)


# **Knowing the dataset**

In [ ]:
#merge two data sets into one table
df_cards = pd.concat([df_cards_train, df_cards_test])

# Shuffle the DataFrame to ensure 'is_fraud=1' records are spread equally
df_cards = df_cards.sample(frac=1, random_state=42).reset_index(drop=True)

df_info(df_cards, "head")

In [ ]:
df_info(df_cards, "columns")

In [ ]:
df_info(df_cards,"info")

In [ ]:
# EDA process
# 1. New column created from local time to indicate weekend or not weekend
# 2. New column of time of a day derived from trans_date_trans_time
# 3. To delete column of transaction hash
# 4. count plot amount against is_fraud
# 5. Encoding on genter(F: 1, M: 0)
# 6. Encoding type of merchant - category
# 7. Statistics.


In [ ]:
#plots
#how many records of fraud in the data set
df_cards['is_fraud'].value_counts()



In [ ]:
#count plots
plt.figure(figsize=(15, 6))
sns.countplot(x='category', data=df_cards[df_cards['is_fraud'] == 1])
plt.xticks(rotation=45)
plt.show()


In [ ]:
#pair plots
columns_for_pairplot = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'is_fraud']
df_sampled = df_cards[columns_for_pairplot].sample(n=50000, random_state=42)
sns.pairplot(df_sampled, hue='is_fraud', diag_kind='kde')

In [ ]:
# Convert 'trans_date_trans_time' to datetime format
df_cards_bak = df_cards.copy()
df_cards['trans_date_trans_time'] = pd.to_datetime(df_cards['trans_date_trans_time'])

# Extract day of the week (Monday=0, Sunday=6)
df_cards['day_of_week_raw'] = df_cards['trans_date_trans_time'].dt.dayofweek

# Initialize 'day_a_week' with 1-7 (Monday=1, Sunday=7)
df_cards['day_a_week'] = df_cards['day_of_week_raw'] + 1

# Get the range of years present in the dataset
min_year = df_cards['trans_date_trans_time'].dt.year.min()
max_year = df_cards['trans_date_trans_time'].dt.year.max()

# Load US national holidays for the relevant years
us_holidays = hol.US(years=range(min_year, max_year + 1))

# Create a boolean column to identify national holidays
df_cards['is_national_holiday'] = df_cards['trans_date_trans_time'].dt.date.isin(us_holidays)

# Reclassify national holidays that fall on a weekday:
# If a day is a national holiday AND it's a weekday (day_a_week is 1-5), set 'day_a_week' to 6.
# Weekends (Saturday=6, Sunday=7) remain as they are.
df_cards.loc[(df_cards['is_national_holiday'] == True) & (df_cards['day_a_week'].isin([1, 2, 3, 4, 5])), 'day_a_week'] = 6

# Drop temporary columns
df_cards = df_cards.drop(columns=['day_of_week_raw'])

# Display the value counts for the new 'day_a_week' column to verify
print(df_cards['day_a_week'].value_counts().sort_index())

df_info(df_cards, "head")

In [ ]:
# New column of time of a day derived from trans_date_trans_time
df_cards['time_of_day'] = df_cards['trans_date_trans_time'].dt.hour

df_info(df_cards, "head")

In [ ]:
df_cards = df_cards.drop(columns=['trans_num'])

In [ ]:
# 1. דגימת הנתונים (קריטי בגלל גודל המאגר)
# כדי שהגרף ייווצר מהר ולא יתקע את המחשב, ניקח מדגם אקראי.
# חשוב להשתמש ב-stratify כדי לשמור על היחס המקורי של ההונאות במדגם.

#df_sampled, _ = train_test_split(df_cards, train_size=100000, stratify=df_cards['is_fraud'], random_state=42)
# 2. הגדרת עיצוב בסיסי לגרף
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))
# 3. יצירת ההיסטוגרמה (Displot)
# 'amt' -> המשתנה בציר X (סכום)
# hue='is_fraud' -> צביעה נפרדת לפי סוג העסקה
# element="step" -> יוצר קווי מתאר חלקים וברורים יותר
# stat="density" -> מנרמל כל קבוצה בנפרד (כדי שנוכל להשוות את צורת ההתפלגות)
# log_scale=(True, False) -> מפעיל סולם לוגריתמי רק על ציר X (כדי לראות סכומי ענק)
g = sns.displot(data=df_cards, x='amt', hue='is_fraud', kind="hist",
                element="step", stat="density", log_scale=(True, False),
                palette=['#1f77b4', '#d62728'], height=6, aspect=1.5)
# 4. כותרות מוסברות היטב
g.set_axis_labels("Transaction Amount (Log Scale)", "Density")
g.fig.suptitle('Distribution of Transaction Amounts: Normal vs. Fraud', fontsize=16, fontweight='bold', y=1.02)
# 5. עדכון המקרא (Legend) שיהיה ברור
new_labels = ['Normal', 'Fraud']
for t, l in zip(g._legend.texts, new_labels): t.set_text(l)
plt.show()

# **Statistical analysis**

In [ ]:
median_no_fraud = df_cards[df_cards['is_fraud'] == 0]['amt'].median()
median_fraud = df_cards[df_cards['is_fraud'] == 1]['amt'].median()
std_no_fraud = df_cards[df_cards['is_fraud'] == 0]['amt'].std()
std_fraud = df_cards[df_cards['is_fraud'] == 1]['amt'].std()

print(f"Median amount for non-fraudulent transactions: ${median_no_fraud:.2f}")
print(f"Median amount for fraudulent transactions: ${median_fraud:.2f}")
print(f"Std for fraudulent transactions: ${std_fraud:.2f}")
print(f"Std for non-fraudulent transactions: ${std_no_fraud:.2f}")

# **Statistical tests.**

In [ ]:
import scipy.stats as stats

# 1. two groups of transaction amount
fraud_amounts = df_cards[df_cards['is_fraud'] == 1]['amt']
normal_amounts = df_cards[df_cards['is_fraud'] == 0]['amt']

# 2. Welch's t-test
t_stat, p_value = stats.ttest_ind(fraud_amounts, normal_amounts, equal_var=False)

# 3. Printing results
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4e}")

# 4. Conclusion
alpha = 0.05
if p_value < alpha:
    print("\nThe difference between the groups is statistically significant")
    print("The mean of the transaction amounts of the fraudent transactions is significantly higher then the mean of the normal transactions.")
else:
    print("\nThe conclusion: no statistical significance")
    print("It cannot be concluded with certainty that the difference between the means is non random")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. מוודאים שהעמודות קיימות: נייצר אותן מחדש כאן ועכשיו
df_cards['day_name'] = pd.to_datetime(df_cards['trans_date_trans_time']).dt.day_name()
df_cards['is_weekend'] = df_cards['day_name'].isin(['Saturday', 'Sunday']).astype(int)

# 2. חישוב אחוזי ההונאה (עכשיו העמודה 'is_weekend' בוודאות קיימת!)
print("--- (Fraud Rate) ---")
fraud_rates = df_cards.groupby('is_weekend')['is_fraud'].mean() * 100
fraud_rates.index = ['Weekday (1-5)', 'Weekend (6-7)']
print(fraud_rates.round(3).astype(str) + '%')
print("\n")

# 3. הכנה למבחן חי-בריבוע: יצירת טבלת צלב
contingency_table = pd.crosstab(df_cards['is_weekend'], df_cards['is_fraud'])

# 4. הרצת המבחן הסטטיסטי (Chi-Square)
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# 5. הדפסת התוצאות
print("--- Chi-Square test ---")
print(f"P-value: {p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print("\nThe conclusion: the result is statistically significant")
    pirnt("The is strong connection between the week day and the posibility of the fraud.")
else:
    print("\nThe conclusion: no statistical significancy.")
    print("\nNo difference found between standard deviation of the fraud midweek and the weekends.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. הפתרון: יצירת עמודה חדשה ואמינה של שם היום ישירות מתוך התאריך
# (וודא ששם עמודת התאריך שלך הוא אכן 'trans_date_trans_time', אם לא תשנה אותו כאן)
df_cards['real_day_name'] = pd.to_datetime(df_cards['trans_date_trans_time']).dt.day_name()

# 2. ספירה כמה פעמים כל יום מופיע בדאטה (value_counts מסדר מהגדול לקטן)
day_counts = df_cards['real_day_name'].value_counts()

print("Total amount of transactions per day:")
print(day_counts)
print("-" * 30)

# 3. בונוס: הצגה ויזואלית של כמות העסקאות לפי ימים מסודרים
plt.figure(figsize=(10, 5))

# נגדיר את הסדר ההגיוני של השבוע כדי שהגרף יהיה קריא
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

sns.countplot(x='real_day_name', data=df_cards, order=days_order, palette='Blues_d')
plt.title('Total Transactions per Day of the Week', fontsize=14)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Number of Transactions', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Encoding gender (F: 1, M: 0)
df_cards['gender'] = df_cards['gender'].map({'F': 1, 'M': 0})

df_info(df_cards, "head")

In [ ]:
# Encoding 'category' using one-hot encoding
df_cards = pd.get_dummies(df_cards, columns=['category'], prefix='category')

df_info(df_cards, "head")

In [ ]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c

    return distance

df_cards['distance'] = df_cards.apply(
    lambda row: haversine_distance(row['lat'], row['long'], row['merch_lat'], row['merch_long']),
    axis=1
)

df_info(df_cards, "head")

In [ ]:
import scipy.stats as stats
import pandas as pd

# 1. פיצול עמודת המרחק לשתי קבוצות: הונאות ועסקאות רגילות
# (אנו משתמשים ב-dropna() למקרה שיש תאים ריקים שיהרסו את החישוב)
fraud_distances = df_cards[df_cards['is_fraud'] == 1]['distance'].dropna()
normal_distances = df_cards[df_cards['is_fraud'] == 0]['distance'].dropna()

# 2. הדפסת חציוני המרחק (במקום ממוצע, בגלל הפיזור של המרחקים)
print("--- מרחק חציוני מהבית לעסק ---")
print(f"עסקאות רגילות: {normal_distances.median():.2f} קילומטרים")
print(f"עסקאות הונאה: {fraud_distances.median():.2f} קילומטרים")
print("-" * 30)

# 3. הרצת מבחן המובהקות Mann-Whitney U
# alternative='two-sided' בודק האם יש הבדל כלשהו בין הקבוצות
stat, p_value = stats.mannwhitneyu(fraud_distances, normal_distances, alternative='two-sided')

# 4. הדפסת התוצאה הסטטיסטית
print("--- תוצאות מבחן Mann-Whitney U ---")
print(f"P-value: {p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print("\nהמסקנה: התוצאה מובהקת סטטיסטית! 🚨")
    print("קיים הבדל מהותי ומוכח בין מרחקי הקנייה של עסקאות רגילות לבין מרחקי הונאות.")
else:
    print("\nהמסקנה: אין מובהקות סטטיסטית.")
    print("לא ניתן לקבוע שהמרחק משפיע בצורה מובהקת על הסיכוי להונאה.")

In [ ]:
average_distance_fraud = df_cards[df_cards['is_fraud'] == 1]['distance'].mean()
print(f"Average distance for fraudulent transactions: {average_distance_fraud:.2f} km")
average_distance_no_fraud = df_cards[df_cards['is_fraud'] == 0]['distance'].mean()
print(f"Average distance for non fraudulent transactions: {average_distance_no_fraud:.2f} km")

In [ ]:
average_distances = df_cards.groupby('is_fraud')['distance'].mean()
print("Average distance for Normal transactions (0): {:.2f} km".format(average_distances[0]))
print("Average distance for Fraud transactions (1): {:.2f} km".format(average_distances[1]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Group data by fraud status and hour, count occurrences, and normalize within each group
hourly_dist = df_cards.groupby(['is_fraud', 'time_of_day']).size().rename('count').reset_index()
total_counts = df_cards.groupby('is_fraud').size()
hourly_dist['percentage'] = hourly_dist.apply(lambda row: (row['count'] / total_counts[row['is_fraud']]) * 100, axis=1)

plt.figure(figsize=(14, 7))
sns.barplot(data=hourly_dist, x='time_of_day', y='percentage', hue='is_fraud', palette=['#1f77b4', '#d62728'])

plt.title('Normalized Transaction Distribution by Hour: Normal vs. Fraud', fontsize=15, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Percentage of Total Transactions in Category (%)', fontsize=12)
plt.legend(title='Transaction Type', labels=['Normal', 'Fraud'])
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. הכנה למבחן: יצירת טבלת צלב (Contingency Table)
# הטבלה סופרת כמה עסקאות רגילות והונאות התבצעו בכל שעה ספציפית ביום
contingency_table_hours = pd.crosstab(df_cards['time_of_day'], df_cards['is_fraud'])

# 2. הרצת המבחן הסטטיסטי (Chi-Square)
chi2, p_value, dof, expected = chi2_contingency(contingency_table_hours)

# 3. הדפסת התוצאות כמו מקצוען
print("--- תוצאות מבחן Chi-Square: שעת העסקה מול הונאות ---")
print(f"P-value: {p_value:.4e}")

# 4. הסקת המסקנה הסטטיסטית
alpha = 0.05
if p_value < alpha:
    print("\nהמסקנה: התוצאה מובהקת סטטיסטית! 🚨")
    print("יש קשר מובהק ומוכח בין שעת ביצוע העסקה לבין הסיכוי שזוהי הונאה.")
    print("(במילים אחרות: גנבים אכן פועלים בשעות שונות מאנשים נורמטיביים)")
else:
    print("\nהמסקנה: אין מובהקות סטטיסטית.")
    print("ההבדלים שראינו בשעות השונות כנראה נובעים ממקריות בדגימה.")

In [ ]:
plt.figure(figsize=(12, 6))
# Filter for fraud transactions and plot the count per hour
sns.countplot(x='day_a_week', data=df_cards[df_cards['is_fraud'] == 1], color='#d62728')
plt.title('Count of Fraudulent Transactions by day of a week', fontsize=14)
plt.xlabel('Day of a week')
plt.ylabel('Number of Frauds')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def get_cc_vendor(cc_num):
    cc_str = str(cc_num)
    if cc_str.startswith('4'):
        return 'Visa'
    elif cc_str.startswith(('51', '52', '53', '54', '55')) or (2221 <= int(cc_str[:4]) <= 2720):
        return 'Mastercard'
    elif cc_str.startswith(('34', '37')):
        return 'Amex'
    elif cc_str.startswith('6011') or cc_str.startswith('65'):
        return 'Discover'
    elif cc_str.startswith(('300', '301', '302', '303', '304', '305', '36', '38')):
        return 'Diners'
    elif cc_str.startswith(('3528', '3589')):
        return 'JCB'
    else:
        return 'Other'

df_cards['cc_vendor'] = df_cards['cc_num'].apply(get_cc_vendor)

print("Credit Card Vendor Distribution:")
display(df_cards['cc_vendor'].value_counts())

df_info(df_cards, 'head')

In [ ]:
# Calculate the fraud rate (percentage) for each vendor
vendor_stats = df_cards.groupby('cc_vendor')['is_fraud'].agg(['count', 'sum'])
vendor_stats['fraud_rate'] = (vendor_stats['sum'] / vendor_stats['count']) * 100
vendor_stats = vendor_stats.sort_values(by='fraud_rate', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=vendor_stats.index, y=vendor_stats['fraud_rate'], palette='viridis', hue=vendor_stats.index, legend=False)

plt.title('Percentage of Fraudulent Transactions by Credit Card Vendor', fontsize=14)
plt.xlabel('Credit Card Vendor')
plt.ylabel('Fraud Rate (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# Filter for fraudulent transactions and plot count by vendor
fraud_by_vendor = df_cards[df_cards['is_fraud'] == 1]['cc_vendor'].value_counts()
sns.barplot(x=fraud_by_vendor.index, y=fraud_by_vendor.values, palette='viridis')

plt.title('Number of Fraudulent Transactions by Credit Card Vendor', fontsize=14)
plt.xlabel('Credit Card Vendor')
plt.ylabel('Count of Frauds')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Calculate fraud rate by gender
gender_fraud_rate = df_cards.groupby('gender')['is_fraud'].mean() * 100

# Map back for labels: 1 is Female, 0 is Male
gender_labels = ['Male (0)', 'Female (1)']

plt.figure(figsize=(8, 6))
sns.barplot(x=gender_fraud_rate.index, y=gender_fraud_rate.values, palette='coolwarm', hue=gender_fraud_rate.index, legend=False)

plt.title('Percentage of Fraudulent Transactions by Gender', fontsize=14)
plt.xlabel('Gender (0=Male, 1=Female)')
plt.ylabel('Fraud Rate (%)')
plt.xticks(ticks=[0, 1], labels=gender_labels)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add text labels on top of bars
for i, val in enumerate(gender_fraud_rate.values):
    plt.text(i, val + 0.01, f"{val:.3f}%", ha='center', fontweight='bold')

plt.show()

In [ ]:
# Calculate the percentage of each gender WITHIN each fraud category
gender_dist = df_cards.groupby('is_fraud')['gender'].value_counts(normalize=True).rename('percentage').reset_index()

# Map gender back to labels
gender_dist['gender_label'] = gender_dist['gender'].map({1: 'Female', 0: 'Male'})
gender_dist['percentage'] *= 100

# Plotting
plt.figure(figsize=(10, 6))
sns.barplot(x='is_fraud', y='percentage', hue='gender_label', data=gender_dist)

plt.title('Gender Distribution within Fraud vs. Normal Transactions', fontsize=14)
plt.xlabel('Transaction Type')
plt.ylabel('Percentage within Category (%)')

# Set descriptive x-axis labels
plt.xticks(ticks=[0, 1], labels=['Normal', 'Fraud'])

plt.legend(title='Gender')

# Add labels on top of bars
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.2f}%',
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'center',
                   xytext = (0, 9),
                   textcoords = 'offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Identify boolean columns
bool_cols = df_cards.select_dtypes(include=['bool']).columns

# Transform boolean columns to integer (1 for True, 0 for False)
df_cards[bool_cols] = df_cards[bool_cols].astype(int)

print(f"Transformed columns: {list(bool_cols)}")
display(df_cards[bool_cols].head())

In [ ]:
# Convert dob to datetime if it's not already
df_cards['dob'] = pd.to_datetime(df_cards['dob'])

# Calculate age: (Transaction Date - Date of Birth)
# We use the total days divided by 365.25 to account for leap years
df_cards['age'] = (df_cards['trans_date_trans_time'] - df_cards['dob']).dt.days // 365

print("Age column created. Statistical summary of age:")
pd.set_option('display.float_format', lambda x: '%2f' %x)
display(df_cards['age'].describe())
display(df_cards[['dob', 'trans_date_trans_time', 'age']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate statistics
mean_age = df_cards['age'].mean()
std_age = df_cards['age'].std()

plt.figure(figsize=(10, 6))
sns.histplot(df_cards['age'], kde=True, color='skyblue')

# Add lines for mean and standard deviation
plt.axvline(mean_age, color='red', linestyle='--', label=f'Mean: {mean_age:.2f}')
plt.axvline(mean_age - std_age, color='green', linestyle=':', label=f'-1 STD: {std_age:.2f}')
plt.axvline(mean_age + std_age, color='green', linestyle=':', label=f'+1 STD: {std_age:.2f}')

plt.title('Age Distribution with Mean and Standard Deviation')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter for fraud cases
df_fraud = df_cards[df_cards['is_fraud'] == 1]

# Calculate statistics for fraud cases
mean_age_fraud = df_fraud['age'].mean()
std_age_fraud = df_fraud['age'].std()

plt.figure(figsize=(10, 6))
sns.histplot(df_fraud['age'], kde=True, color='#d62728')

# Add lines for mean and standard deviation
plt.axvline(mean_age_fraud, color='black', linestyle='--', label=f'Mean (Fraud): {mean_age_fraud:.2f}')
plt.axvline(mean_age_fraud - std_age_fraud, color='blue', linestyle=':', label=f'-1 STD: {std_age_fraud:.2f}')
plt.axvline(mean_age_fraud + std_age_fraud, color='blue', linestyle=':', label=f'+1 STD: {std_age_fraud:.2f}')

plt.title('Age Distribution of Fraud Victims with Mean and STD')
plt.xlabel('Age')
plt.ylabel('Frequency of Fraud')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# יצירת גרף KDE (התפלגות צפיפות) להשוואת אוכלוסיות
# common_norm=False דואג שכל קבוצה תנורמל בנפרד ל-100%
sns.kdeplot(data=df_cards, x='age', hue='is_fraud', fill=True,
            common_norm=False, palette=['#1f77b4', '#d62728'], alpha=0.5)

plt.title('Normalized Age Distribution: Normal vs. Fraud Transactions', fontsize=16, fontweight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Density (Normalized)', fontsize=12)
plt.legend(title='Transaction Type', labels=['Fraud', 'Normal'])
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

In [ ]:
df_cards2 = df_cards.copy()
# Dropping non-predictive or redundant numeric columns for a cleaner heatmap
df_cards2 = df_cards2.drop(columns=['merch_long', 'merch_lat', 'lat', 'long', 'unix_time', 'cc_num'])

numeric_df = df_cards2.select_dtypes(include=['number'])
if 'Unnamed: 0' in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=['Unnamed: 0'])

# Plotting the correlations with annotations to see the actual numbers
plt.figure(figsize=(12, 10))
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap: Blue indicates low/negative correlation, Red indicates positive')
plt.show()

print("Top correlations with is_fraud:")
display(correlation_matrix['is_fraud'].sort_values(ascending=False).to_frame())

# **Label Encoding**

In [ ]:
def simplify_category(cat):
    cat = cat.lower()
    if '_net' in cat:
        return 'net'
    elif any(word in cat for word in ['grocery', 'gas', 'kids', 'pets']):
        return 'home_supplies'
    elif any(word in cat for word in ['food', 'dining', 'entertainment']):
        return 'lifestyle'
    elif 'travel' in cat:
        return 'travel'
    else:
        return 'other'

# Note: Since 'category' was already one-hot encoded in 'df_cards',
# we'll use 'df_cards_bak' (your backup) or reconstruct from existing columns if needed.
# Assuming we want to apply this to a fresh copy or the main dataframe:

if 'category' in df_cards_bak.columns:
    df_cards['category_grouped'] = df_cards_bak['category'].apply(simplify_category)
    print("New grouped category distribution:")
    print(df_cards['category_grouped'].value_counts())
else:
    print("Original 'category' column not found in backup. Please ensure df_cards_bak is available.")

### Statistics


### T-tests for Numerical Features vs. Fraud Status

Here, we perform independent samples T-tests (specifically, Welch's t-test which does not assume equal variances) for several numerical features to see if their mean values differ significantly between fraudulent (`is_fraud=1`) and non-fraudulent (`is_fraud=0`) transactions.

In [ ]:
import scipy.stats as stats
import pandas as pd

numerical_features_for_ttest = [
    'amt', 'zip', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long',
    'day_a_week', 'time_of_day', 'distance', 'age'
]

alpha = 0.05

print("Performing T-tests for numerical features comparing fraudulent vs. non-fraudulent transactions:\n")

for feature in numerical_features_for_ttest:
    if feature in df_cards.columns:
        print(f"--- Testing '{feature}' vs. 'is_fraud' ---")

        # Separate the numerical feature values for fraud and non-fraud groups
        values_fraud = df_cards[df_cards['is_fraud'] == 1][feature].dropna()
        values_non_fraud = df_cards[df_cards['is_fraud'] == 0][feature].dropna()

        # Ensure both groups have enough data for a T-test
        if len(values_fraud) > 1 and len(values_non_fraud) > 1:
            # Perform Welch's t-test (does not assume equal variances)
            t_stat, p_value = stats.ttest_ind(values_fraud, values_non_fraud, equal_var=False)

            print(f"  Mean for Non-Fraudulent ({feature}): {values_non_fraud.mean():.2f}")
            print(f"  Mean for Fraudulent ({feature}): {values_fraud.mean():.2f}")
            print(f"  T-statistic: {t_stat:.4f}")
            print(f"  P-value: {p_value:.4e}")

            if p_value < alpha:
                print(f"  Conclusion: There is a statistically significant difference in the mean of '{feature}' between fraudulent and non-fraudulent transactions.")
            else:
                print(f"  Conclusion: No statistically significant difference in the mean of '{feature}' found between fraudulent and non-fraudulent transactions.")
        else:
            print(f"  Not enough data in one or both groups for feature '{feature}' to perform T-test.")
        print("-" * 50)
    else:
        print(f"Feature '{feature}' not found in df_cards. Skipping.\n" + "-" * 50)

print("\n--- T-test Analysis Complete ---")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

print("Performing Chi-squared tests for association between categorical features and 'is_fraud':\n")

alpha = 0.05

# Identify categorical columns from the DataFrame
categorical_features = []

# Add binary categorical features (already encoded as 0/1)
if 'gender' in df_cards.columns:
    categorical_features.append('gender')
if 'is_national_holiday' in df_cards.columns:
    categorical_features.append('is_national_holiday')

# Add one-hot encoded 'category_' columns dynamically
for col in df_cards.columns:
    if col.startswith('category_') and col != 'category_grouped':
        categorical_features.append(col)

# Add other multi-level categorical/ordinal features that were processed
if 'day_a_week' in df_cards.columns:
    categorical_features.append('day_a_week')
if 'time_of_day' in df_cards.columns:
    categorical_features.append('time_of_day')
if 'cc_vendor' in df_cards.columns:
    categorical_features.append('cc_vendor')
if 'category_grouped' in df_cards.columns:
    categorical_features.append('category_grouped')
if 'real_day_name' in df_cards.columns:
    categorical_features.append('real_day_name')

# Use a set to remove potential duplicates and iterate
for feature in sorted(list(set(categorical_features))):
    if feature in df_cards.columns:
        print(f"--- Testing '{feature}' vs. 'is_fraud' ---")
        # Create a contingency table
        contingency_table = pd.crosstab(df_cards[feature], df_cards['is_fraud'])

        # Perform Chi-squared test
        try:
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            print(f"  Chi2 Statistic: {chi2:.4f}")
            print(f"  P-value: {p_value:.4e}")

            if p_value < alpha:
                print(f"  Conclusion: There is a statistically significant association between '{feature}' and 'is_fraud'.")
            else:
                print(f"  Conclusion: No statistically significant association found between '{feature}' and 'is_fraud'.")
        except ValueError as e:
            print(f"  Error performing Chi-squared test for '{feature}': {e}. This might occur if there are too few observations or categories with zero counts.")
        print("-" * 50)
    else:
        print(f"Feature '{feature}' not found in df_cards. Skipping.")

print("\n--- Chi-squared Test Summary Complete ---")

### ANOVA (Analysis of Variance) for Transaction Amount by Grouped Category

ANOVA helps determine if there are any statistically significant differences between the means of three or more independent groups. Here, we'll use it to assess if the mean transaction amount (`amt`) varies significantly across the different `category_grouped` types, separately for normal and fraudulent transactions.

In [ ]:
import scipy.stats as stats

# --- ANOVA for Non-Fraudulent Transactions ---
print("--- ANOVA for Non-Fraudulent Transactions ---")

# Filter for non-fraudulent transactions
df_normal = df_cards[df_cards['is_fraud'] == 0]

# Create a list of transaction amounts for each category group
# Ensure there are at least two groups to compare
groups_normal = [df_normal['amt'][df_normal['category_grouped'] == cat].values for cat in df_normal['category_grouped'].unique() if len(df_normal['amt'][df_normal['category_grouped'] == cat].values) > 0]

# Perform one-way ANOVA
if len(groups_normal) > 1:
    f_stat_normal, p_value_normal = stats.f_oneway(*groups_normal)
    print(f"F-statistic (Normal): {f_stat_normal:.4f}")
    print(f"P-value (Normal): {p_value_normal:.4e}")

    alpha = 0.05
    if p_value_normal < alpha:
        print("Conclusion (Normal): There is a statistically significant difference in mean transaction amounts across grouped categories for non-fraudulent transactions.")
    else:
        print("Conclusion (Normal): No statistically significant difference in mean transaction amounts across grouped categories for non-fraudulent transactions.")
else:
    print("Not enough category groups for ANOVA in non-fraudulent transactions.")

print("\n--- ANOVA for Fraudulent Transactions ---")

# --- ANOVA for Fraudulent Transactions ---
# Filter for fraudulent transactions
df_fraud = df_cards[df_cards['is_fraud'] == 1]

# Create a list of transaction amounts for each category group
groups_fraud = [df_fraud['amt'][df_fraud['category_grouped'] == cat].values for cat in df_fraud['category_grouped'].unique() if len(df_fraud['amt'][df_fraud['category_grouped'] == cat].values) > 0]

# Perform one-way ANOVA
if len(groups_fraud) > 1:
    f_stat_fraud, p_value_fraud = stats.f_oneway(*groups_fraud)
    print(f"F-statistic (Fraud): {f_stat_fraud:.4f}")
    print(f"P-value (Fraud): {p_value_fraud:.4e}")

    alpha = 0.05
    if p_value_fraud < alpha:
        print("Conclusion (Fraud): There is a statistically significant difference in mean transaction amounts across grouped categories for fraudulent transactions.")
    else:
        print("Conclusion (Fraud): No statistically significant difference in mean transaction amounts across grouped categories for fraudulent transactions.")
else:
    print("Not enough category groups for ANOVA in fraudulent transactions.")

### Spearman test

In [ ]:
import pandas as pd

# Select only numerical columns for correlation analysis
# Exclude 'Unnamed: 0' and 'cc_num' as they are identifiers and not meaningful for correlation
numerical_cols = df_cards.select_dtypes(include=['number']).columns.tolist()
if 'Unnamed: 0' in numerical_cols:
    numerical_cols.remove('Unnamed: 0')
if 'cc_num' in numerical_cols:
    numerical_cols.remove('cc_num')

df_numerical = df_cards[numerical_cols]

# Calculate the Spearman correlation matrix
spearman_corr = df_numerical.corr(method='spearman')

print("Spearman Correlation Matrix:")
display(spearman_corr)

print("\nTop correlations with 'is_fraud' (Spearman):")
display(spearman_corr['is_fraud'].sort_values(ascending=False).to_frame())

### Conclusions from EDA
*  **Redundant features:** According to statistics evaluation following features are less significant: distance, gender, national_holiday, city_pop, merch_long, long.
*  **Transaction Amount:** There is a significant correlation between the transaction amount and fraud status.
*  **Holidays:** Fraudulent transactions show a correlation with national holidays.
*  **Outliers:** Since transaction amount is a primary indicator of fraud, removing outliers would lead to losing critical data, which would negatively affect prediction accuracy.
*  **Model Selection:** The lack of linear correlation between various features suggests that tree-based models (like Random Forest or XGBoost) would be most suitable for this classification task.
